# BCI Closed-Loop Evidence

This notebook exercises SC-NeuroCore's deterministic closed-loop BCI template from raw waveform windows to AER payloads, rate decoding, implant-emulator feedback, telemetry, and reference HIL manifests.

## Evidence Boundary

This notebook uses synthetic waveform windows and an in-process implant emulator only. It does not claim clinical safety, physical implant access, stimulation approval, human-subject validation, or hardware-in-the-loop timing closure. Real BCI validation requires approved protocols, physical acquisition/stimulation hardware, captured latency traces, safety interlocks, and external review.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np

from sc_neurocore.interfaces.bci_closed_loop import (
    ClosedLoopBCIConfig,
    ClosedLoopBCITemplate,
    ImplantEmulator,
    RateSpikeDecoder,
)
from sc_neurocore.interfaces.bci_hil_manifest import (
    available_bci_hil_profiles,
    build_bci_hil_reference_manifest,
    create_bci_hil_template,
)

REPO = Path.cwd()
if not (REPO / "src").exists():
    REPO = Path.cwd().parent

waveform = np.zeros((96, 4), dtype=np.float32)
waveform[12, 0] = -25.0
waveform[40, 1] = -30.0
waveform[72, 3] = -35.0

template = ClosedLoopBCITemplate(
    ClosedLoopBCIConfig(
        n_channels=4,
        sampling_rate_hz=30_000,
        threshold_sigma=4.0,
        snippet_samples=16,
        waveform_mode="spike",
        feedback_gain=1.0,
        max_feedback=1.0,
    )
)
result = template.process_window(waveform, window_start_us=500)
closed_loop_summary = {
    "detected_spikes": int(result.spike_raster.sum()),
    "waveform_spikes_detected": result.waveform.n_spikes_detected,
    "aer_events": result.aer.n_events,
    "aer_magic": result.aer_payload[:4].decode("ascii"),
    "decoded_rates": [float(value) for value in result.decoded_rates.tolist()],
    "feedback": {
        "values": list(result.feedback.values),
        "timestamp_us": result.feedback.timestamp_us,
        "active_count": result.feedback.active_count,
    },
    "telemetry_total_ticks": result.telemetry["total_ticks"],
}
assert closed_loop_summary["detected_spikes"] == 3
assert closed_loop_summary["aer_magic"] == "AERX"
closed_loop_summary

In [ ]:
decoder = RateSpikeDecoder(sampling_rate_hz=1_000)
raster = np.array([[1, 0], [0, 1], [1, 0], [0, 0]], dtype=np.int8)
rates = decoder.decode(raster)

emulator = ImplantEmulator(gain=2.0, max_feedback=1.0)
frame = emulator.apply_feedback(np.array([0.25, 2.0, -2.0], dtype=np.float32), timestamp_us=10)
decoder_sink_summary = {
    "rates": [float(value) for value in rates.tolist()],
    "clipped_feedback": list(frame.values),
    "recorded_frames": len(emulator.frames),
}
assert np.allclose(rates, np.array([500.0, 250.0]))
assert frame.values == (0.5, 1.0, -1.0)
decoder_sink_summary

In [ ]:
profiles = available_bci_hil_profiles()
reference_manifest = build_bci_hil_reference_manifest("probe_384ch")
reference_template = create_bci_hil_template("probe_384ch")
manifest_summary = {
    "profile_ids": [profile.profile_id for profile in profiles],
    "selected_profile": reference_manifest["profile"]["profile_id"],
    "schema_version": reference_manifest["schema_version"],
    "hardware_required": reference_manifest["profile"]["safety_contract"]["hardware_required"],
    "no_stimulation_without_sink_override": reference_manifest["profile"]["safety_contract"]["no_stimulation_without_sink_override"],
    "template_channels": reference_template.config.n_channels,
    "pipeline_steps": reference_manifest["profile"]["pipeline_steps"],
}
assert "probe_384ch" in manifest_summary["profile_ids"]
assert manifest_summary["hardware_required"] is False
assert manifest_summary["no_stimulation_without_sink_override"] is True
manifest_summary

In [ ]:
try:
    template.process_window(np.zeros((16, 3), dtype=np.float32))
except ValueError as exc:
    mismatch_refusal = str(exc)
else:
    raise AssertionError("channel mismatch was accepted")

try:
    template.process_window(np.zeros(4, dtype=np.float32))
except ValueError as exc:
    shape_refusal = str(exc)
else:
    raise AssertionError("non-matrix waveform was accepted")

guardrail_summary = {
    "channel_mismatch_refusal": mismatch_refusal,
    "shape_refusal": shape_refusal,
}
assert "expected 4" in mismatch_refusal
assert "shape" in shape_refusal
guardrail_summary

In [ ]:
manifest = {
    "schema_version": "sc-neurocore.bci-closed-loop-evidence.v1",
    "closed_loop_summary": closed_loop_summary,
    "decoder_sink_summary": decoder_sink_summary,
    "reference_manifest_summary": manifest_summary,
    "guardrails": guardrail_summary,
    "evidence_boundary": "Synthetic waveform plus implant emulator only; no clinical, stimulation, physical implant, or HIL timing claim.",
}
manifest